In [2]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT # to create databases
from psycopg2.extras import execute_batch

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [ ]:
df = pd.read_csv('Flight_data.csv')


In [14]:
def generate_flight_data(original_df, num_new_rows=150):

    
    arrival_stats = original_df.groupby('Arrival City').agg({ #agg по городу и данные по ним. Некий профиль города или статистика по городу. Далее для генерации похожих данных следующие шаги:
        'Departure City': lambda x: x.mode()[0] if not x.mode().empty else None, #mode() мода - самое частое отправление для прибытий сюда, берем первое самое частое [0].
        'Origin': lambda x: x.mode()[0] if not x.mode().empty else None, #Из какого аэропорта чаще всего
        'Destination': lambda x: x.mode()[0] if not x.mode().empty else None, #Куда прибытие чаще всего
        'Route': lambda x: x.mode()[0] if not x.mode().empty else None, #Та же логика
        'Ticket Price': 'mean',
        'Competitor Price': 'mean',
        'Flight Duration': 'mean',          #Здесь среднее значение численных переменных
        'Demand': 'mean',
        'Profitability': 'mean'
    }).reset_index()
    
    new_data = []

    for i in range(num_new_rows):
        arrival_row = arrival_stats.sample(1).iloc[0] #Рандомный город прибытия

        max_id = original_df['Customer ID'].max() #Послений уникальный номер
        new_id = max_id + i + 1 #Другие новые уникальные номера потом

        start_date = datetime(2023, 1, 1)
        end_date = datetime(2023, 12, 31)
        random_date = start_date + timedelta( #start_dat + разница во времени
            seconds=random.randint(0, int((end_date - start_date).total_seconds()))) #генерируем кол-во секунд от 0 до разницы


        new_row = { #словарь с нновой записью
                'Departure City': arrival_row['Departure City'],
                'Arrival City': arrival_row['Arrival City'], #по которому группировали 
                'Departure Date': random_date.strftime('%Y-%m-%d %H:%M:%S'),
                'Flight Duration': arrival_row['Flight Duration'] * random.uniform(0.9, 1.1), #ср. длительность полета +- 10 %
                'Delay Minutes': random.randint(0, 180), 
                'Customer ID': new_id,
                'Name': generate_random_name(),
                'Booking Class': random.choice(['Economy', 'Business', 'First']),
                'Frequent Flyer Status': random.choice(['Silver', 'Gold', 'Platinum']),
                'Route': arrival_row['Route'], #типичный маршрут из профиля
                'Ticket Price': arrival_row['Ticket Price'] * random.uniform(0.8, 1.2),
                'Competitor Price': arrival_row['Competitor Price'] * random.uniform(0.8, 1.2),  
                'Demand': arrival_row['Demand'] * random.uniform(0.8, 1.2),
                'Origin': arrival_row['Origin'],
                'Destination': arrival_row['Destination'],
                'Profitability': arrival_row['Profitability'] * random.uniform(0.8, 1.2),
                'Loyalty Points': random.randint(0, 5000),
                'Churned': random.choice([True, False])
            }
        
        new_data.append(new_row)
    return pd.DataFrame(new_data)
    
def generate_random_name():
    first_names = ['George', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 
                'Linda', 'William', 'Elizabeth', 'David', 'Barbara', 'Richard', 'Susan',
                'Joseph', 'Jessica', 'Thomas', 'Sarah', 'Charles', 'Karen']
    
    last_names = ['Rafaelyan', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller',
                'Davis', 'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez',
                'Wilson', 'Anderson', 'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin']
    
    return f"{random.choice(first_names)} {random.choice(last_names)}" #рандомный выбор
new_df = generate_flight_data(df, 150)
combined_df = pd.concat([df, new_df], ignore_index=True)
output_path = r"C:\Users\Daria\Documents\hse\py\py_project\Flight_data.csv"
combined_df.to_csv(output_path, index=False)

print("\nПоследние 5 сгенерированных строк:")
print(new_df['Customer ID'].tail(2))
print(len(combined_df))



Последние 5 сгенерированных строк:
148    10010
149    10011
Name: Customer ID, dtype: int64
250
